# 05 - ML Pipeline Orchestration (Task DAG)

This notebook covers:
1. Understanding the configurable ML pipeline architecture
2. Deploying the DAG to Snowflake
3. Running the pipeline with custom configuration
4. Monitoring task execution and results

**Pipeline Stages:**
```
PREPARE_DATA -> REFRESH_FEATURES -> TRAIN_MODEL -> EVALUATE_MODEL -> QUALITY_GATE -> [DEPLOY_MODEL | NOTIFY_ALERT]
```

**Key Design:**
- All parameters passed via DAG `config={}` (hyperparams, thresholds, deployment settings)
- Each task reads only the config keys it needs via `TaskContext.get_task_graph_config()`
- Same pipeline code works for dev and prod by swapping config values
- Quality gate uses configurable thresholds to auto-approve or reject model versions

In [ ]:
import sys
sys.path.insert(0, "../source")
from snowpark_session import create_snowpark_session
from config import DATABASE, SCHEMA, WAREHOUSE, PIPELINE_CONFIG

session = create_snowpark_session()
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()

print("Default pipeline configuration:")
for k, v in PIPELINE_CONFIG.items():
    print(f"  {k}: {v}")

## Step 1: Deploy the Pipeline DAG

This creates a Snowflake Task Graph with all pipeline stages. The root task is suspended by default (manual trigger for dev).

In [ ]:
from pipeline.ml_pipeline_dag import deploy_dag, run_dag

# Deploy with default config
deploy_dag(session=session)

## Step 2: Verify DAG Structure

In [ ]:
# Show all tasks in the pipeline
session.sql(f"""
    SHOW TASKS LIKE '%FRAUD_DETECTION%' IN SCHEMA {DATABASE}.{SCHEMA}
""").show()

## Step 3: Run the Pipeline

Trigger a manual execution. The pipeline will:
1. Validate source data
2. Refresh Feature Store Dynamic Tables
3. Train XGBoost on compute pool (ML Job)
4. Evaluate metrics against quality thresholds
5. Deploy to SPCS service if gate passes, or alert if it fails

In [ ]:
# Trigger the pipeline
run_dag(session=session)

## Step 4: Monitor Execution

In [ ]:
# Check task history (re-run to see progress)
session.sql(f"""
    SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME, 
           DATEDIFF('second', SCHEDULED_TIME, COMPLETED_TIME) AS DURATION_SEC,
           ERROR_MESSAGE
    FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY())
    WHERE DATABASE_NAME = '{DATABASE}'
    ORDER BY SCHEDULED_TIME DESC
    LIMIT 20
""").show()

## Step 5: Re-deploy with Custom Config

Change pipeline parameters without modifying code. Example: tighter thresholds for production.

In [ ]:
# Example: deploy with stricter thresholds (prod-like)
prod_config = {
    **PIPELINE_CONFIG,
    "min_auc_roc": "0.90",
    "min_precision": "0.80",
    "min_recall": "0.70",
    "n_estimators": "300",
    "max_depth": "8",
    "model_version": "V2",
}

# Uncomment to redeploy with prod config:
# deploy_dag(session=session, config=prod_config)
# run_dag(session=session, config=prod_config)
print("Prod config ready. Uncomment above to deploy with stricter thresholds.")